# 12.4 · BERT 与编码器模型 / BERT & Encoder Models

> **课程定位 / Where this fits**
> 第 4 课，**Part 12**。Transformer 三大用法的第一种：**编码器**。
> Lesson 4, **Part 12**. The first of three Transformer uses: the **encoder**.
>
> Transformer 有三种用法(12.3 结尾提过)：**编码器**(双向理解)、**解码器**(自回归生成)、**编码-解码**(转换)。**BERT**(2018)用**编码器**+**双向**自注意力，靠**掩码语言模型(MLM)** 在海量无标注文本上预训练，再微调到下游——一举刷新了几乎所有"理解类"任务(分类、问答抽取、NER、检索)。本课**从零实现 MLM 预训练**，看模型如何学会"完形填空"，并讲清**双向 vs 单向**、**预训练→微调**范式。
> The Transformer has three uses (noted at the end of 12.3): **encoder** (bidirectional understanding), **decoder** (autoregressive generation), **encoder-decoder** (transduction). **BERT** (2018) uses the **encoder** + **bidirectional** self-attention, pretrained on massive unlabeled text via **masked language modeling (MLM)**, then fine-tuned downstream — smashing nearly all "understanding" tasks (classification, extractive QA, NER, retrieval). We **implement MLM pretraining from scratch**, watch the model learn "fill-in-the-blank," and explain **bidirectional vs unidirectional** and the **pretrain→fine-tune** paradigm.
>
> 💼 **实战/面试视角**："MLM 是什么 / BERT 为什么双向 / 预训练任务(MLM+NSP) / BERT vs GPT / 怎么微调" 是 NLP 岗必考。
> 💼 **Practical/interview angle:** "what is MLM / why BERT is bidirectional / pretraining tasks (MLM+NSP) / BERT vs GPT / how to fine-tune" — NLP must-knows.

> 📐 **符号约定 / Notation**
> - [MASK] —— MLM 中被遮住、要预测的占位符 / the masked placeholder to predict
> - [CLS] —— 句首特殊 token, 其输出代表整句 / sentence-level token

> 💡 **面试相关 / Interview-relevant**
> - "MLM(掩码语言模型)怎么做"（出镜率 ★★★★★）
> - "BERT 双向 vs GPT 单向的区别与各自适合的任务"（★★★★★）
> - "BERT 的两个预训练任务 MLM+NSP"（★★★★）
> - "[CLS]/[SEP] token 的作用"（★★★）
> - "预训练→微调范式"（★★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 BERT = Transformer 编码器 + 双向。
   Understand BERT = Transformer encoder + bidirectional.
2. **从零实现 MLM 预训练**(掩码并预测)。
   Implement MLM pretraining from scratch (mask and predict).
3. 看到模型用**双向上下文**完形填空。
   See the model fill blanks using **bidirectional context**.
4. 理解预训练→微调范式与 BERT 应用。
   Understand pretrain→fine-tune and BERT applications.

## 目录 / TOC
1. [BERT：双向编码器 ⭐](#1)
2. [掩码语言模型 MLM（从零）⭐](#2)
3. [双向的威力：完形填空 ⭐](#3)
4. [预训练→微调 + BERT vs GPT + 小结 ⭐](#4)


<a id="1"></a>
## 1. BERT：双向编码器 ⭐ / BERT: Bidirectional Encoder

**BERT = Bidirectional Encoder Representations from Transformers**。核心两点：
**BERT = Bidirectional Encoder Representations from Transformers.** Two key points:
- **只用编码器**：堆叠 Transformer 编码器块(就是 12.3 搭的那种, 自注意力**不加因果掩码**)。
  **Encoder-only:** stacks Transformer encoder blocks (like 12.3's, self-attention **without a causal mask**).
- **双向(bidirectional)**：每个 token 能**同时看到左边和右边**的所有词。这对**理解**任务至关重要——判断 "bank" 是"银行"还是"河岸"，要看整句两侧。
  **Bidirectional:** each token sees **all words on both sides simultaneously**. Crucial for **understanding** — to disambiguate "bank," you need both sides of the sentence.

**但有个矛盾**：传统语言模型靠"预测下一个词"训练，可如果每个词都能看到右边，"预测下一个词"就**作弊**了(答案就在右边)。BERT 的解法是 **MLM**：随机**遮住**一些词，让模型用**两侧上下文**预测被遮的词——既能双向、又有自监督训练信号。
**But there's a tension:** classic LMs train by "predict the next word," yet if each word sees its right side, that's **cheating** (the answer is on the right). BERT's solution is **MLM**: randomly **mask** some words and predict them from **both-side context** — bidirectional *and* a self-supervised signal.

我们用一个**结构化合成语言**演示(便于看清 MLM 学到了什么)：句子形如 `the <主语> <动词> the <宾语>`，主语是动物、宾语是物品。
We demo with a **structured synthetic language** (so we can see what MLM learns): sentences like `the <subject> <verb> the <object>`, subjects are animals, objects are things.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, math, time
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid")
torch.manual_seed(0); np.random.seed(0)

SUBJ = ["dog","cat","bird","fish","lion","shark"]        # 主语:动物 / subjects: animals
VERB = ["eats","sees","chases","likes"]                  # 动词 / verbs
OBJ  = ["food","water","worm","bone","meat"]             # 宾语:物品 / objects
vocab = ["[PAD]","[MASK]","the"] + SUBJ + VERB + OBJ
w2i = {w:i for i,w in enumerate(vocab)}; V = len(vocab); MASK = w2i["[MASK]"]
def sentence(): return ["the", np.random.choice(SUBJ), np.random.choice(VERB), "the", np.random.choice(OBJ)]
def encode(s): return [w2i[w] for w in s]
print(f"词表({V}): {vocab}")
print(f"句子模板: the <动物> <动词> the <物品>; 例: {' '.join(sentence())}")
print("MLM 目标: 随机遮住一个词(换成[MASK]), 让模型用两侧上下文猜出它")


<a id="2"></a>
## 2. 掩码语言模型 MLM（从零）⭐ / Masked Language Model From Scratch

**MLM 流程**：随机选 ~15% 的 token，把它们换成 `[MASK]`，让模型(Transformer 编码器 + 一个输出层)**预测原词**，用交叉熵训练。**只在被遮的位置算损失**(其它位置不算)。
**MLM procedure:** randomly pick ~15% of tokens, replace with `[MASK]`, and have the model (Transformer encoder + an output head) **predict the originals**, trained with cross-entropy. **Loss only at masked positions** (others ignored).

> 细节(面试)：实际 BERT 对选中的 15% 中，80% 换 `[MASK]`、10% 换随机词、10% 不变——为了缓解"预训练有[MASK]、微调没有[MASK]"的不匹配。这里简化为全部换 `[MASK]`。
> Detail (interview): real BERT, of the 15% chosen, replaces 80% with `[MASK]`, 10% with a random word, 10% unchanged — to reduce the "pretrain has [MASK], fine-tune doesn't" mismatch. We simplify to all `[MASK]`.

下面复用 12.3 的 Transformer 编码器块，从零训练 MLM。
We reuse 12.3's Transformer encoder block and train MLM from scratch.


In [ ]:
# --- 复用 12.3 的从零 Transformer 编码器 / reuse the from-scratch encoder ---
def sdpa(q,k,v):
    d=q.size(-1); a=F.softmax(q@k.transpose(-2,-1)/math.sqrt(d), -1); return a@v, a
class MHA(nn.Module):
    def __init__(s,D,H): super().__init__(); s.H=H; s.d=D//H; s.qkv=nn.Linear(D,3*D); s.o=nn.Linear(D,D)
    def forward(s,x):
        B,T,D=x.shape; qkv=s.qkv(x).reshape(B,T,3,s.H,s.d).permute(2,0,3,1,4)
        o,a=sdpa(qkv[0],qkv[1],qkv[2]); return s.o(o.transpose(1,2).reshape(B,T,D)), a
class Block(nn.Module):
    def __init__(s,D,H): super().__init__(); s.a=MHA(D,H); s.n1=nn.LayerNorm(D); s.n2=nn.LayerNorm(D); s.ff=nn.Sequential(nn.Linear(D,4*D),nn.ReLU(),nn.Linear(4*D,D))
    def forward(s,x): h,a=s.a(s.n1(x)); x=x+h; return x+s.ff(s.n2(x)), a
class BertEncoder(nn.Module):
    def __init__(s,V,D=64,H=4,L=2,maxlen=8):
        super().__init__(); s.emb=nn.Embedding(V,D); s.pos=nn.Parameter(torch.randn(1,maxlen,D)*0.02)
        s.blocks=nn.ModuleList([Block(D,H) for _ in range(L)])
    def forward(s,x):
        h=s.emb(x)+s.pos[:,:x.size(1)]; A=[]
        for b in s.blocks: h,a=b(h); A.append(a)        # 无因果掩码 → 双向 / no causal mask → bidirectional
        return h, A

torch.manual_seed(0); enc = BertEncoder(V); mlm_head = nn.Linear(64, V)
opt = torch.optim.Adam(list(enc.parameters())+list(mlm_head.parameters()), 3e-3)
ce = nn.CrossEntropyLoss(ignore_index=-100)              # -100 的位置不算损失 / ignore non-masked
losses = []
t0 = time.time()
for step in range(800):
    batch = torch.tensor([encode(sentence()) for _ in range(128)])
    inp = batch.clone(); labels = torch.full_like(batch, -100)
    for i in range(len(batch)):
        j = np.random.randint(1, 5)                      # 随机遮一个内容位置 / mask one content position
        labels[i, j] = batch[i, j]; inp[i, j] = MASK     # 记录答案 + 替换为[MASK] / record answer + mask
    h, _ = enc(inp); logits = mlm_head(h)
    loss = ce(logits.reshape(-1, V), labels.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
fig, ax = plt.subplots(figsize=(7,3.2)); ax.plot(losses); ax.set_xlabel("训练步"); ax.set_ylabel("MLM 损失")
ax.set_title(f"MLM 预训练损失下降 ({time.time()-t0:.0f}s)"); plt.tight_layout(); plt.show()
print(f"MLM 预训练完成, 损失 {losses[0]:.2f} → {losses[-1]:.2f}")
print("模型学会: 用两侧上下文预测被[MASK]遮住的词(完形填空)")


<a id="3"></a>
## 3. 双向的威力：完形填空 ⭐ / Bidirectionality: Cloze Test

训练好的 MLM 就是个**完形填空高手**。关键验证：它填空时**同时利用左右两侧**的上下文。我们遮住不同位置看它能否填出**类别正确**的词。
The trained MLM is a **cloze master**. Key check: it uses **both left and right** context to fill. We mask different positions and see if it fills with the **right category**.


In [ ]:
def fill(tokens, pos, k=4):
    """遮住 pos 位置, 返回 top-k 预测词 / mask position `pos`, return top-k predictions."""
    ids = torch.tensor([encode(tokens)]); ids[0, pos] = MASK
    with torch.no_grad(): logits = mlm_head(enc(ids)[0])[0, pos]
    return [vocab[i] for i in logits.topk(k).indices.tolist()]

print("完形填空(模型用两侧上下文预测被遮位置):")
print(f"  the [MASK] eats the bone   → {fill(['the','dog','eats','the','bone'], 1)}   (应是动物/主语)")
print(f"  the dog [MASK] the bone    → {fill(['the','dog','eats','the','bone'], 2)}   (应是动词)")
print(f"  the cat sees the [MASK]    → {fill(['the','cat','sees','the','food'], 4)}   (应是物品/宾语)")

# 定量验证: 遮主语位, 预测落在'动物'类别的比例 / quantitative: masked-subject → animal-category rate
correct = 0; N = 500
for _ in range(N):
    s = sentence(); ids = torch.tensor([encode(s)]); ids[0,1] = MASK
    with torch.no_grad(): pred = mlm_head(enc(ids)[0])[0,1].argmax().item()
    correct += vocab[pred] in SUBJ
print(f"\n遮住主语位 → 预测为'动物类'词的比例 = {correct/N:.3f}")
print("模型靠'前面是the、后面是动词'(两侧线索)判断该填主语 → 这就是双向理解的价值")
print("对比 GPT(单向): 只能用左侧上下文, 看不到右边; 故 BERT 更适合'理解'类任务")


<a id="4"></a>
## 4. 预训练→微调 + BERT vs GPT + 小结 ⭐ / Pretrain→Fine-tune, BERT vs GPT

**BERT 的范式 = 预训练 + 微调**(呼应 9.13 迁移学习、10.11 自监督)：
**BERT's paradigm = pretrain + fine-tune** (echoing 9.13 transfer learning, 10.11 self-supervised):
1. **预训练**：在海量**无标注**文本上做 MLM(+NSP)，学到通用语言表示。(只做一次, 很贵)
   **Pretrain:** MLM (+NSP) on massive **unlabeled** text → general language representations. (Once, expensive.)
2. **微调**：在预训练好的编码器上**加一个小任务头**(如分类头), 用**少量标注数据**训练整个模型几轮即可。
   **Fine-tune:** add a small **task head** (e.g. classifier) on the pretrained encoder, train on **little labeled data** for a few epochs.

**[CLS] token**(面试)：BERT 在句首加一个特殊 `[CLS]` token，预训练后**它的输出向量代表整句**，微调分类时就接在它上面(像 ViT 的 class token, 10.9)。句对任务用 `[SEP]` 分隔两句。
**[CLS] token:** BERT prepends a special `[CLS]`; its output **represents the whole sentence**, used for classification heads (like ViT's class token, 10.9). Sentence pairs are separated by `[SEP]`.

**第二个预训练任务 NSP(下一句预测)**：判断两句是否相邻，帮助学句子关系。后续 RoBERTa 发现 NSP 帮助不大、去掉了它。
**Second pretraining task NSP (Next Sentence Prediction):** predict whether two sentences are adjacent, to learn sentence relations. RoBERTa later found NSP unhelpful and dropped it.

**BERT vs GPT(超高频对比)**：
**BERT vs GPT (super-frequent comparison):**

| | BERT(编码器) | GPT(解码器) |
|---|---|---|
| 注意力 | **双向**(看两侧) | **单向/因果**(只看左) |
| 预训练 | MLM 完形填空 | 预测下一个词 |
| 强项 | **理解**: 分类/抽取/NER/检索 | **生成**: 续写/对话/创作 |
| 用法 | 微调到下游任务 | 提示/生成(也可微调) |

```
BERT: Transformer编码器 + 双向自注意力(无因果掩码); 适合理解类任务
MLM: 遮住~15%词(80%[MASK]/10%随机/10%不变), 用两侧上下文预测; 只在遮住位算损失
双向价值: 同时用左右上下文消歧/理解; GPT单向只能看左
[CLS]: 句首token, 输出代表整句用于分类; [SEP]分隔句对
预训练任务: MLM + NSP(下一句预测; RoBERTa去掉NSP)
范式: 无标注大数据预训练(贵,一次) → 少量标注微调(快); 这是NLP迁移学习
BERT(理解) vs GPT(生成); 后续 RoBERTa/ALBERT/ELECTRA/DeBERTa 等改进
```

### 💡 面试速查 / Interview cheat-sheet
1. **MLM**: 遮词→用两侧上下文预测; 只在遮住位算损失; 实现双向自监督。
   MLM: mask words → predict from both sides; loss only at masked positions; bidirectional self-supervision.
2. **双向 vs 单向**: BERT看两侧(理解), GPT只看左(生成)。
   Bidirectional vs unidirectional: BERT both sides (understanding), GPT left-only (generation).
3. **预训练任务**: MLM + NSP(RoBERTa去NSP)。
   Pretraining: MLM + NSP (RoBERTa drops NSP).
4. **[CLS]**: 代表整句, 微调分类接在其输出。
   [CLS]: represents the sentence; classification head sits on it.
5. **范式**: 大数据预训练 → 小数据微调; 适合分类/QA/NER/检索。
   Paradigm: pretrain big → fine-tune small; great for classification/QA/NER/retrieval.

### 下一节 / Next
**12.5 GPT 与解码器模型**——Transformer 的**解码器**用法。GPT 用**因果(单向)自注意力**做**自回归生成**：逐词预测下一个词。我们会**从零搭一个字符级 GPT 并训练它生成文本**，这是理解 ChatGPT 等生成式大模型的基础。
**12.5 GPT & Decoder Models** — the Transformer **decoder**. GPT uses **causal (unidirectional) self-attention** for **autoregressive generation**: predict the next token. We'll **build a char-level GPT from scratch and train it to generate text** — the basis for understanding ChatGPT-style generative LLMs.
